# Прототип мульти-агентной системы для командировок

Минимальный пример на LangGraph: менеджер делегирует запрос поисковику и возвращает ответ.
RAG-пайплайн описан на уровне комментариев (чанкинг, эмбеддинг, реранжинг).


In [ ]:
!pip -q install langgraph langchain-core

In [ ]:
from typing import TypedDict, List, Optional
from langgraph.graph import StateGraph, END
from langchain_core.messages import AIMessage

# Пример данных, которые обычно приходят из RAG/поиска
MOCK_FLIGHTS = [
    "SU100: NYC -> LAX, 2026-03-10, $320",
    "AA205: NYC -> LAX, 2026-03-10, $410",
]

def mock_search(request: str) -> str:
    # Здесь в реальном решении:
    # 1) Чанкинг документов политики командировок
    # 2) Эмбеддинг чанков и запись в Vector DB
    # 3) Ретривер + реранкер для лучших фрагментов
    # 4) Агент использует фрагменты как контекст
    return " | ".join(MOCK_FLIGHTS)

class GraphState(TypedDict):
    request: str
    messages: List[AIMessage]
    search_result: Optional[str]
    final: Optional[str]

def manager(state: GraphState) -> GraphState:
    if state.get("search_result") is None:
        return {
            **state,
            "messages": state["messages"] + [AIMessage(content="Делегирую поиск билетов...")],
        }
    answer = (
        "Подобраны варианты перелетов: " + state["search_result"] +
        ". При необходимости уточните бюджет и предпочтения."
    )
    return {
        **state,
        "final": answer,
        "messages": state["messages"] + [AIMessage(content=answer)],
    }

def searcher(state: GraphState) -> GraphState:
    result = mock_search(state["request"])
    return {
        **state,
        "search_result": result,
        "messages": state["messages"] + [AIMessage(content="Поиск завершен.")],
    }

def route(state: GraphState):
    if state.get("search_result") is None:
        return "searcher"
    return END

graph = StateGraph(GraphState)
graph.add_node("manager", manager)
graph.add_node("searcher", searcher)
graph.set_entry_point("manager")
graph.add_conditional_edges("manager", route, {"searcher": "searcher", END: END})
graph.add_edge("searcher", "manager")
app = graph.compile()


In [ ]:
result = app.invoke({"request": "Нужна командировка NYC -> LAX на 2026-03-10", "messages": [], "search_result": None, "final": None})
print(result["final"])


Подобраны варианты перелетов: SU100: NYC -> LAX, 2026-03-10, $320 | AA205: NYC -> LAX, 2026-03-10, $410. При необходимости уточните бюджет и предпочтения.